# Values

`Values` is GTSAM's heterogeneous, key-indexed container for nonlinear variable assignments. It stores typed manifold values while providing uniform insertion, update, retraction, and local-coordinate operations.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/nonlinear/doc/Values.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [ ]:
import gtsam
import numpy as np
from gtsam.symbol_shorthand import L, V, X

## Inserting values

Each key may appear once and has a fixed stored type. `insert()` rejects duplicate keys; use `update()` or `insert_or_assign()` when replacement is intended.

In [ ]:
values = gtsam.Values()
values.insert(X(0), gtsam.Pose2(1.0, 2.0, 0.1))
values.insert(L(0), np.array([4.0, 5.0]))
values.insert(gtsam.symbol("d", 0), 2.5)
print("size:", values.size(), "dimension:", values.dim())
print("keys:", [gtsam.DefaultKeyFormatter(k) for k in values.keys()])

## Typed retrieval and updates

Python exposes typed accessors such as `atPose2()`, `atPoint2()`, and `atDouble()`. `exists()` checks a key before retrieval, and `erase()` removes it.

In [ ]:
print("pose:", values.atPose2(X(0)))
print("landmark:", values.atPoint2(L(0)))

values.update(X(0), gtsam.Pose2(1.1, 2.0, 0.1))
assert values.exists(X(0))
print("updated pose:", values.atPose2(X(0)))

## Retraction and local coordinates

`zeroVectors()` creates a zero `VectorValues` tangent with the correct dimensions. To specify an increment, insert one tangent vector per key. `retract(delta)` applies it, while `localCoordinates(other)` computes the displacement between compatible containers.

In [ ]:
zero = values.zeroVectors()
print("zero tangent norm:", zero.norm())

delta = gtsam.VectorValues()
delta.insert(X(0), np.array([0.01, -0.02, 0.03]))
delta.insert(L(0), np.array([0.1, -0.1]))
delta.insert(gtsam.symbol("d", 0), np.array([0.2]))

perturbed = values.retract(delta)
recovered = values.localCoordinates(perturbed)
for key in values.keys():
    np.testing.assert_allclose(recovered.at(key), delta.at(key), atol=1e-8)

## Combining containers

`insert(other_values)` merges disjoint keys, `swap()` exchanges contents, and `clear()` removes everything. Container equality is manifold-aware through `equals()`; it is not just Python object identity.

## Source

[`Values.h`](../Values.h)

## AI assistance caveat

AI was used to help draft this documentation, and inaccuracies could be present.